In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from datetime import datetime
import time
import pandas as pd
import os

CSV_FILE = "停車位.csv"

def main():
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")

    driver = webdriver.Chrome(options=chrome_options)

    try:
        driver.get("https://www.parkinginfo.ntpc.gov.tw/parkinginfo/Public/CertifyQuery.aspx")
        time.sleep(2)

        # Search "秀朗國小"
        search_input = driver.find_element(By.ID, "txtEParkingLotName")
        search_input.clear()
        search_input.send_keys("秀朗國小")

        # Click search
        driver.find_element(By.ID, "btnQueryList").click()
        time.sleep(3)

        # Extract remaining spots
        spot_element = driver.find_element(By.CSS_SELECTOR, 'div[data-th="剩餘車位"] span')
        remaining_spots = spot_element.text.strip()

        # Get current time
        now_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        # DataFrame
        new_row = pd.DataFrame([{
            "時間": now_time,
            "停車場名稱": "秀朗國小地下停車場",
            "剩餘車位": remaining_spots
        }])

        # Create or Append
        if os.path.exists(CSV_FILE):
            df = pd.read_csv(CSV_FILE)
            df = pd.concat([df, new_row], ignore_index=True)
        else:
            df = new_row

        df.to_csv(CSV_FILE, index=False, encoding="utf-8-sig")

        print(f"{now_time} - 剩餘車位: {remaining_spots} 已存入 {CSV_FILE}")

    finally:
        driver.quit()

while True:
    main()
    time.sleep(1200)


2025-08-14 21:38:49 - 剩餘車位: 84 已存入 停車位.csv
